# Lab 6: Gaussian Mixtures and the EM Algorithm

In this lab we implement the EM algorithm for Gaussian mixtures from scratch, first in 1-D on the lengths of bolts made by two machines, then in d dimensions, and compare it with k-means. A real dataset (Iris) is homework.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from itertools import permutations
from matplotlib.patches import Ellipse
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.datasets import load_iris
plt.style.use("seaborn-v0_8")

## Helpers

The functions below are given. Run the cells and move on.

**`load_lengths`** returns the lengths (in mm) of 381 bolts made by two machines (the counts of the lecture's table "Two regimes in one sample").

In [ ]:
def load_lengths():
    """Return x, a (381,) array of lengths in mm, rebuilt from the table of counts of the lecture."""
    lengths = [82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 98]
    counts = [5, 3, 12, 36, 55, 45, 21, 13, 15, 34, 59, 48, 16, 12, 6, 1]
    # every length is repeated as many times as it was counted
    return np.repeat(lengths, counts).astype(float)

**`make_data`** simulates points from a mixture of 3 Gaussians in 2-D, as in the lecture ("Simulating mixture"): first the cluster $z_i$ of the point, then the point $x_i$ from the Gaussian of that cluster.

In [ ]:
def make_data(n=300, seed=0):
    """Return X, a (n, 2) array of points, and z, the (n,) array of their true clusters (0, 1 or 2)."""
    rng = np.random.RandomState(seed)

    def stretched(angle, length, width):
        # covariance of a Gaussian stretched in the direction angle: standard deviation length along it, width across it
        R = np.array([[np.cos(angle), -np.sin(angle)], [np.sin(angle), np.cos(angle)]])
        return R @ np.diag([length ** 2, width ** 2]) @ R.T

    p = [0.4, 0.3, 0.3]
    mus = [np.array([0., 0.]), np.array([1.5, 3.]), np.array([5., 1.])]
    Sigmas = [stretched(0.6, 2.2, 0.4), stretched(0.6, 2.2, 0.4), 0.64 * np.eye(2)]

    # first the cluster of every point, then the point from the Gaussian of its cluster
    z = rng.choice(3, size=n, p=p)
    X = np.array([rng.multivariate_normal(mus[k], Sigmas[k]) for k in z])
    return X, z

**`start_labels`** gives the random start of EM: $K$ random points are the starting centres, and every point gets the label of the closest centre. It works for 1-D data of shape $(n,)$ and for data of shape $(n, d)$.

In [ ]:
def start_labels(X, K, seed=0):
    """Return a (n, K) array of 0/1 labels: entry (i, k) is 1 if centre k is the closest to point i."""
    rng = np.random.RandomState(seed)
    X_2d = X.reshape(len(X), -1)                       # (n, d), also when X is 1-D
    # K different points of the data as centres
    candidates = np.unique(X_2d, axis=0)
    centres = candidates[rng.choice(len(candidates), K, replace=False)]
    # squared distance of every point to every centre, then the closest centre
    distances = np.sum((X_2d[:, None, :] - centres[None, :, :]) ** 2, axis=2)
    return np.eye(K)[distances.argmin(axis=1)]

**`match_labels`** renames the found clusters so that they match the true groups as well as possible: EM and k-means number their clusters in any order, so without it the same cluster would get a different colour in every plot. **`share_correct`** returns the share of points in the cluster of their own group.

In [ ]:
def match_labels(labels, truth):
    """Rename the clusters (labels) so that as many points as possible get the number of their true group (truth)."""
    K = max(labels.max(), truth.max()) + 1
    # try every renaming of the K clusters, keep the one with the most matches
    best = max(permutations(range(K)), key=lambda names: np.sum(np.array(names)[labels] == truth))
    return np.array(best)[labels]


def share_correct(labels, truth):
    """Share of points whose (renamed) cluster is their true group."""
    return np.mean(match_labels(labels, truth) == truth)

**`plot_clusters`** draws 2-D points coloured by their labels and, if given, the ellipse of every cluster (the area within 2 standard deviations of its mean).

In [ ]:
def plot_clusters(ax, X, labels, title, means=None, covariances=None):
    """Draw the (n, 2) points X on ax, coloured by labels, and an ellipse per cluster if means and covariances are given."""
    ax.scatter(X[:, 0], X[:, 1], c=labels, cmap="viridis", s=12)
    if means is not None:
        for mu, Sigma in zip(means, covariances):
            # the axes of the ellipse are the eigenvectors of Sigma, of length 2 standard deviations on each side
            values, vectors = np.linalg.eigh(Sigma)
            angle = np.degrees(np.arctan2(vectors[1, 1], vectors[0, 1]))
            ax.add_patch(Ellipse(mu, 4 * np.sqrt(values[1]), 4 * np.sqrt(values[0]), angle=angle,
                                 fill=False, color="red", linewidth=2))
    ax.set_title(title)

## 1. EM in 1-D

**The data.** The lengths $x_1, \dots, x_n$ of $n = 381$ bolts from one box, made by two machines of a factory. Each machine is set to its own length, but the machine that made each bolt, its **machine of origin**, was not recorded (it is the hidden label $z_i$).

**The clusters** are the machines: cluster $k$ = the bolts made by machine $k$. Each cluster is described by one Gaussian bell (in ML terms, a **mixture component**: scikit-learn calls their number `n_components`).

In [ ]:
x = load_lengths()
print("x:", x.shape, "| smallest:", x.min(), "| largest:", x.max())

plt.figure(figsize=(8, 4))
plt.hist(x, bins=np.arange(81.5, 99.5, 1), color="lightgray", edgecolor="gray")
plt.xlabel("length (mm)")
plt.ylabel("bolts")
plt.title("Lengths of bolts: two peaks, no machine label")
plt.show()

**What we look for.** For every cluster $k = 1, \dots, K$:
- $p_k$, the **weight**: the **percentage** of the bolts in cluster $k$, $p_k = n_k / n$, where $n_k$ is the **number** of bolts in cluster $k$ (30 bolts out of $n = 100$ give $p_k = 0.3$; the $p_k$ add up to 1);
- $\mu_k$, the **mean**: the average length in cluster $k$;
- $\sigma_k^2$, the **variance**: how spread the lengths are around $\mu_k$.

And for every bolt $i$:
- $\gamma_{ik}$, the **responsibility**: the probability that bolt $i$ comes from cluster $k$ (for every $i$, the $\gamma_{ik}$ add up to 1).

**The model.** Cluster $k$ is a Gaussian bell $f_k$, and the data is a mix of the $K$ bells, each with its weight:
$$
f_k(x) = \frac{1}{\sqrt{2\pi\sigma_k^2}} \exp\Big(-\frac{(x - \mu_k)^2}{2\sigma_k^2}\Big), \qquad
f(x) = p_1 f_1(x) + \dots + p_K f_K(x).
$$

**How to find them: EM.** If we knew the cluster of every bolt, it would be easy: $p_k$ is the share of bolts in cluster $k$, $\mu_k$ their mean and $\sigma_k^2$ their variance. We do not know it, so EM alternates two steps:

1. **Start** (given, `start_labels`): a first guess: $K$ random bolts as centres, every bolt in the cluster of the closest centre ($\gamma_{ik} = 1$ for that cluster, 0 for the others), then one M-step.
2. **E-step (Expectation):** with $p$, $\mu$, $\sigma$ fixed, compute $\gamma$ for every bolt:
$$
\gamma_{ik} = \frac{p_k \, f_k(x_i)}{p_1 f_1(x_i) + \dots + p_K f_K(x_i)} .
$$
3. **M-step (Maximization):** with $\gamma$ fixed, compute $p$, $\mu$, $\sigma$, where bolt $i$ counts for $\gamma_{ik}$ in cluster $k$:
$$
n_k = \sum_{i} \gamma_{ik}, \qquad
p_k = \frac{n_k}{n}, \qquad
\mu_k = \frac{1}{n_k} \sum_{i} \gamma_{ik} \, x_i, \qquad
\sigma_k^2 = \frac{1}{n_k} \sum_{i} \gamma_{ik} \, (x_i - \mu_k)^2 .
$$
   Here $n_k$ is a soft count: a bolt that is 60% in cluster 1 and 40% in cluster 2 adds 0.6 to $n_1$ and 0.4 to $n_2$.
4. **Repeat** steps 2 and 3 until the **log-likelihood**, which measures how well the mixture fits the data, stops going up (by less than `tol`):
$$
\ell = \sum_{i=1}^{n} \log f(x_i) .
$$

The names: $\gamma_{ik}$ is the **expected value** of the hidden label (1 if bolt $i$ comes from cluster $k$, 0 if not), and the M-step formulas **maximize** the log-likelihood for these $\gamma$. The log-likelihood goes up at every iteration.

In NumPy: `X` has shape $(n,)$, the densities and $\gamma$ have shape $(n, K)$, and `p_`, `mu_`, `sigma2_` have shape $(K,)$.

**Exercise 1.** Complete `GMM1D`: `density`, `e_step`, `m_step`, `log_likelihood` and `fit`.

In [ ]:
class GMM1D:
    """Mixture of K Gaussians in 1-D, fitted with the EM algorithm."""

    def __init__(self, K, n_iter=500, tol=1e-6, seed=0):
        self.K = K                  # number of clusters
        self.n_iter = n_iter        # largest number of EM iterations
        self.tol = tol              # stop when the log-likelihood goes up by less than tol
        self.seed = seed            # seed of the random start
        self.p_ = None              # (K,) weights p_k
        self.mu_ = None             # (K,) means mu_k
        self.sigma2_ = None         # (K,) variances sigma_k^2
        self.loglik_history_ = []   # log-likelihood after every iteration

    def density(self, X):
        """(n, K) array: entry (i, k) is f_k(x_i), the normal density of cluster k at point i."""
        # TODO
        # f_k(x) = exp(-(x - mu_k)^2 / (2 sigma_k^2)) / sqrt(2 pi sigma_k^2), for all points and clusters at once:
        # X[:, None] has shape (n, 1) and self.mu_, self.sigma2_ have shape (K,), so X[:, None] - self.mu_ is (n, K)

    def e_step(self, X):
        """E-step (Expectation): (n, K) array of the responsibilities gamma_ik."""
        # TODO
        # Step 1: p_k f_k(x_i) for all i and k: self.density(X) * self.p_
        # Step 2: divide every row by its sum (np.sum with axis=1, keepdims=True), so that every row sums to 1

    def m_step(self, X, gamma):
        """M-step (Maximization): update p_, mu_ and sigma2_ from the (n, K) responsibilities gamma."""
        # TODO
        # Step 1: n_k, the sum of gamma over the points (axis=0)
        # Step 2: self.p_ = n_k / n, with n = len(X)
        # Step 3: self.mu_ = (sum_i gamma_ik x_i) / n_k: np.sum(gamma * X[:, None], axis=0) / n_k
        # Step 4: self.sigma2_ = (sum_i gamma_ik (x_i - mu_k)^2) / n_k, with the new self.mu_

    def log_likelihood(self, X):
        """Log-likelihood of the data: sum_i log(sum_k p_k f_k(x_i))."""
        # TODO
        # sum_k p_k f_k(x_i) for every point i is self.density(X) @ self.p_

    def fit(self, X):
        """Fit p_, mu_ and sigma2_ to the data X with EM."""
        # TODO
        # Step 1: random start: self.m_step(X, start_labels(X, self.K, self.seed))
        # Step 2: self.loglik_history_ = [self.log_likelihood(X)]
        # Step 3: repeat at most self.n_iter times:
        #     a) E-step (expectation), then M-step (maximization): self.m_step(X, self.e_step(X))
        #     b) append self.log_likelihood(X) to self.loglik_history_
        #     c) stop (break) if the log-likelihood went up by less than self.tol
        return self

    def predict(self, X):
        """Hard labels: for every point, the cluster with the largest responsibility."""
        return self.e_step(X).argmax(axis=1)

Fit a mixture of $K = 2$ Gaussians to the lengths:

In [ ]:
model_1d = GMM1D(K=2)
model_1d.fit(x)

history = np.array(model_1d.loglik_history_)
print("Log-likelihood went up at every iteration:", bool(np.all(np.diff(history) >= -1e-9)))
print(f"Iterations: {len(history) - 1}, final log-likelihood: {history[-1]:.2f}")
for k in range(2):
    print(f"Cluster {k + 1}: p = {model_1d.p_[k]:.2f}, mu = {model_1d.mu_[k]:.2f} mm, sigma = {np.sqrt(model_1d.sigma2_[k]):.2f} mm")

plt.figure(figsize=(8, 4))
plt.plot(history, marker="o")
plt.xlabel("iteration")
plt.ylabel("log-likelihood")
plt.title("EM: the log-likelihood goes up at every iteration")
plt.show()

The fitted mixture on the histogram, and the responsibilities along the axis:

In [ ]:
grid = np.linspace(80, 100, 400)
densities = model_1d.density(grid)

plt.figure(figsize=(8, 4))
plt.hist(x, bins=np.arange(81.5, 99.5, 1), density=True, color="lightgray", edgecolor="gray", label="data")
for k in range(2):
    plt.plot(grid, model_1d.p_[k] * densities[:, k], label=f"cluster {k + 1}: p_k f_k(x)")
plt.plot(grid, densities @ model_1d.p_, color="black", label="mixture f(x)")
plt.xlabel("length (mm)")
plt.legend()
plt.show()

gamma_grid = model_1d.e_step(grid)
plt.figure(figsize=(8, 3))
for k in range(2):
    plt.plot(grid, gamma_grid[:, k], label=f"cluster {k + 1}")
plt.xlabel("length (mm)")
plt.ylabel("responsibility")
plt.title("gamma_k(x): the probability that a bolt of length x comes from cluster k")
plt.legend()
plt.show()

EM finds the two machines: two bells at 86.1 mm and 92.3 mm, with standard deviations of about 1.5 mm and weights 0.49 and 0.51. The log-likelihood goes up at every iteration, but not at a constant speed: here it almost stalls between iterations 8 and 19 before it climbs again, which is why we stop only when it stops increasing. The responsibilities cross at about 89 mm: a shorter bolt most likely comes from the first machine, a longer one from the second, and near 89 mm the model is unsure.

## 2. EM in d dimensions

Every point now has $d$ coordinates, $x_i \in \mathbb{R}^d$ (here $d = 2$). Only two things change:
- the mean $\mu_k$ is a vector of $d$ numbers;
- the variance $\sigma_k^2$ becomes a $d \times d$ **covariance matrix** $\Sigma_k$: the variances of the $d$ features on the diagonal, and the covariances between features off the diagonal (they give the tilt). A cluster is now an **ellipse** instead of a bell.

The bell is the 1-D formula with $\Sigma_k$ in place of $\sigma_k^2$: the **determinant** $|\Sigma_k|$ plays the role of the variance $\sigma_k^2$ in front, and the **inverse** $\Sigma_k^{-1}$ replaces dividing by $\sigma_k^2$:
$$
f_k(x) = \frac{1}{\sqrt{(2\pi)^d \, |\Sigma_k|}} \exp\Big(-\frac{1}{2} (x - \mu_k)^T \Sigma_k^{-1} (x - \mu_k)\Big) .
$$

The term $(x - \mu_k)^T \Sigma_k^{-1} (x - \mu_k)$, `squared_distance` in the code, is the squared distance from $x$ to the centre $\mu_k$, divided by the spread: in 1-D it is $(x - \mu_k)^2 / \sigma_k^2$.

In the M-step, $n_k$, $p_k$ and $\mu_k$ keep their formulas ($\mu_k$ is now a vector of $d$ means), and the variance becomes the weighted covariance:
$$
n_k = \sum_{i} \gamma_{ik}, \qquad
p_k = \frac{n_k}{n}, \qquad
\mu_k = \frac{1}{n_k} \sum_{i} \gamma_{ik} \, x_i, \qquad
\Sigma_k = \frac{1}{n_k} \sum_{i} \gamma_{ik} \, (x_i - \mu_k)(x_i - \mu_k)^T + \varepsilon I .
$$
The tiny $\varepsilon I$ ($\varepsilon = 10^{-6}$, $I$ the identity) stops a cluster from shrinking onto a single point (lecture: "Guarding against collapse").

The E-step, the log-likelihood and `fit` do not change: `GMM` inherits them from `GMM1D`, and only `density` and `m_step` change.

In NumPy: `X` has shape $(n, d)$, `mu_` has shape $(K, d)$, and `Sigma_` has shape $(K, d, d)$. The gaps `diff = X - mu_k` are an $(n, d)$ array whose row $i$ is $(x_i - \mu_k)^T$.

**Exercise 2.** Complete `GMM`: `density` and `m_step`.

In [ ]:
class GMM(GMM1D):
    """Mixture of K Gaussians in d dimensions, fitted with the EM algorithm.
    mu_ is a (K, d) array, and the variances are replaced by the covariance matrices Sigma_, a (K, d, d) array."""

    def density(self, X):
        """(n, K) array: entry (i, k) is f_k(x_i), the normal density of cluster k at point i."""
        # TODO
        # Step 1: n, d = X.shape, and densities = np.zeros((n, self.K))
        # Step 2: for every cluster k:
        #     a) diff = X - self.mu_[k], an (n, d) array: row i is (x_i - mu_k)^T
        #     b) squared_distance = (x_i - mu_k)^T Sigma_k^-1 (x_i - mu_k) for every point:
        #        with inv = np.linalg.inv(self.Sigma_[k]), row i of diff @ inv is (x_i - mu_k)^T Sigma_k^-1,
        #        and the last product with (x_i - mu_k) is a dot product, row by row: np.sum((diff @ inv) * diff, axis=1)
        #     c) densities[:, k] = exp(-squared_distance / 2) / sqrt((2 pi)^d |Sigma_k|)   (np.linalg.det)
        # Step 3: return densities

    def m_step(self, X, gamma):
        """M-step (Maximization): update p_, mu_ and Sigma_ from the (n, K) responsibilities gamma."""
        # TODO
        # Step 1: n, d = X.shape, then n_k and self.p_ as in 1-D
        # Step 2: self.mu_ = np.zeros((self.K, d)) and self.Sigma_ = np.zeros((self.K, d, d)), then for every cluster k:
        #     a) mu_k = (sum_i gamma_ik x_i) / n_k: np.sum(gamma[:, k, None] * X, axis=0) / n_k[k]
        #     b) diff = X - self.mu_[k], an (n, d) array: row i is (x_i - mu_k)^T
        #     c) Sigma_k = (sum_i gamma_ik (x_i - mu_k)(x_i - mu_k)^T) / n_k + 1e-6 I:
        #        the sum is the matrix product diff.T @ (gamma[:, k, None] * diff), and I is np.eye(d)

The data: 300 points from 3 Gaussians, two of them long and thin, side by side. As for the bolts, the Gaussian a point was drawn from is its origin $z_i$ (`z`): the mixture never sees it, we only use it to check the clusters.

In [ ]:
X, z = make_data()
print("X:", X.shape, "| points per true cluster:", np.bincount(z))

fig, ax = plt.subplots(figsize=(6, 5))
plot_clusters(ax, X, z, "The true clusters")
plt.show()

Fit a mixture of $K = 3$ Gaussians, and compare it with k-means:

In [ ]:
model = GMM(K=3)
model.fit(X)

history = np.array(model.loglik_history_)
print("Log-likelihood went up at every iteration:", bool(np.all(np.diff(history) >= -1e-9)))
print(f"Iterations: {len(history) - 1}, final log-likelihood: {history[-1]:.2f}")

# the same data, clustered twice; clusters renamed to match the true clusters, so the colours match
labels_kmeans = match_labels(KMeans(n_clusters=3, n_init=10, random_state=0).fit_predict(X), z)
labels_gmm = match_labels(model.predict(X), z)
print(f"Points in their true cluster: k-means {share_correct(labels_kmeans, z):.0%}, GMM {share_correct(labels_gmm, z):.0%}")

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
plot_clusters(axes[0], X, z, "The true clusters")
plot_clusters(axes[1], X, labels_kmeans, "k-means")
plot_clusters(axes[2], X, labels_gmm, "Gaussian mixture", model.mu_, model.Sigma_)
plt.show()

k-means gives every point to its closest centre, so its borders are straight lines and it cuts the two long, thin clusters across: 81% of the points are in their true cluster. The mixture gives every cluster its own ellipse (a shape, not just a centre) and finds the three clusters: 96%.

**Soft labels.** The responsibilities say how sure the model is about every point:

In [ ]:
gamma = model.e_step(X)
confidence = gamma.max(axis=1)
print("Unsure points (largest responsibility below 0.9):", np.sum(confidence < 0.9), "of", len(X))

plt.figure(figsize=(6, 5))
plt.scatter(X[:, 0], X[:, 1], c=confidence, cmap="viridis", s=12)
plt.colorbar(label="largest responsibility")
plt.title("How sure the mixture is about every point")
plt.show()

17 of the 300 points have a largest responsibility below 0.9: they lie where two ellipses overlap. The mixture says it is unsure there, while k-means only gives a hard label.

**Check with scikit-learn.** `GaussianMixture` runs the same EM (`score` is the mean log-likelihood per point):

In [ ]:
sk_model = GaussianMixture(n_components=3, random_state=0).fit(X)
print(f"Log-likelihood, ours:    {model.log_likelihood(X):.2f}")
print(f"Log-likelihood, sklearn: {sk_model.score(X) * len(X):.2f}")

## Homework: real data, Iris

150 flowers of 3 species, described by their petal length and petal width (lecture: "Example on real data"). The mixture does not see the species: we only use them to check the clusters.

In [ ]:
iris = load_iris()
X_iris = iris.data[:, 2:4]   # petal length and petal width
species = iris.target

model_iris = GMM(K=3).fit(X_iris)
labels_iris = match_labels(model_iris.predict(X_iris), species)   # same numbers (colours) as the species
confidence_iris = model_iris.e_step(X_iris).max(axis=1)
print(f"Flowers in the cluster of their own species: {share_correct(labels_iris, species):.0%}")
print("Unsure flowers (largest responsibility below 0.9):", np.sum(confidence_iris < 0.9), "of", len(X_iris))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plot_clusters(axes[0], X_iris, species, "The true species")
plot_clusters(axes[1], X_iris, labels_iris, "The mixture's clusters", model_iris.mu_, model_iris.Sigma_)
for ax in axes:
    ax.set_xlabel("petal length")
    ax.set_ylabel("petal width")
plt.show()

98% of the flowers land in the cluster of their own species, and 17 of the 150 have a largest responsibility below 0.9 (lecture: 97% and 17). All the mistakes and all the unsure flowers are between the two species that overlap (versicolor and virginica), while setosa is perfectly separated. EM numbers its clusters in any order, so `match_labels` renames them to match the species before plotting (lecture: "Mixture relabeled").